In [1]:
import joblib
import numpy as np
import pandas as pd
import sys
sys.path.append(".")

from state_coords import STATE_COORDS

DATA_PATH = "../outputs/cleaned_data.csv"
FACTORY_PATH = "../data/factories.csv"
MODEL_PATH = "../models/best_lead_time_model.joblib"
OUT_PATH = "../outputs/reassignment_recommendations.csv"

df = pd.read_csv(DATA_PATH)
factories = pd.read_csv(FACTORY_PATH)
# bundle = joblib.load(MODEL_PATH)
# pipeline = bundle["pipeline"]
bundle = joblib.load("../models/random_forest_model.joblib")
pipeline = bundle["pipeline"]

print(bundle["model_name"])

Random Forest


In [2]:
# rf_bundle = joblib.load("../models/random_forest_model.joblib")
# rf_pipeline = rf_bundle["pipeline"]
# print(rf_bundle["model_name"])

In [3]:
# rf_model = rf_pipeline.named_steps["model"]
# rf_preprocessor = rf_pipeline.named_steps["prep"]
# rf_feature_names = rf_preprocessor.get_feature_names_out()

# importances = pd.Series(rf_model.feature_importances_, index=rf_feature_names)
# factory_importances = importances[importances.index.str.contains("Factory")].sort_values(ascending=False)
# factory_importances

In [4]:
# importances.sort_values(ascending=False).head(10)

In [5]:
from geo_utils import haversine

In [6]:
def simulate_product_region(product_name, division, region, ship_mode, units, current_factory):
    dest_lat = df[df["Region"] == region]["Dest_Lat"].mean()
    dest_lon = df[df["Region"] == region]["Dest_Lon"].mean()

    rows = []
    for _, frow in factories.iterrows():
        factory = frow["Factory"]
        dist = haversine(frow["Latitude"], frow["Longitude"], dest_lat, dest_lon)

        X = pd.DataFrame([{
            "Product Name": product_name,
            "Factory": factory,
            "Region": region,
            "Ship Mode": ship_mode,
            "Division": division,
            "Shipping_Distance_KM": dist,
            "Units": units,
        }])
        pred_lead_time = pipeline.predict(X)[0]

        rows.append({
            "Factory": factory,
            "Distance_KM": round(dist, 1),
            "Predicted_Lead_Time": round(pred_lead_time, 2),
            "Is_Current": factory == current_factory,
        })

    result = pd.DataFrame(rows).sort_values("Predicted_Lead_Time")
    return result.reset_index(drop=True)

In [7]:
example = simulate_product_region(
    product_name="Wonka Bar - Milk Chocolate",
    division="Chocolate",
    region="Pacific",
    ship_mode="Standard Class",
    units=5,
    current_factory="Wicked Choccy's",
)
example

,Factory,Distance_KM,Predicted_Lead_Time,Is_Current
0,Lot's O' Nuts,870.4,6.47,False
1,Sugar Shack,2065.6,6.58,False
2,Secret Factory,2372.6,6.74,False
3,Wicked Choccy's,3429.5,6.94,True
4,The Other Factory,2541.1,7.00,False


In [8]:
combos = (
    df.groupby(["Product Name", "Division", "Region", "Ship Mode", "Factory"])
    .agg(
        avg_units=("Units", "mean"),
        avg_profit_margin=("Profit_Margin", "mean"),
        order_count=("Order ID", "count"),
    )
    .reset_index()
)

print(f"Total combinations to simulate: {len(combos)}")
combos.head()

Total combinations to simulate: 141


,Product Name,Division,Region,Ship Mode,Factory,avg_units,avg_profit_margin,order_count
0,Everlasting Gobstopper,Sugar,Gulf,Standard Class,Secret Factory,4.0,0.8,1
1,Everlasting Gobstopper,Sugar,Interior,Standard Class,Secret Factory,3.0,0.8,1
2,Fizzy Lifting Drinks,Sugar,Atlantic,First Class,Sugar Shack,4.0,0.6,1
3,Fizzy Lifting Drinks,Sugar,Atlantic,Standard Class,Sugar Shack,5.0,0.6,1
4,Fizzy Lifting Drinks,Sugar,Gulf,Standard Class,Sugar Shack,4.0,0.6,1


In [9]:
recommendations = []

for _, row in combos.iterrows():
    sim = simulate_product_region(
        product_name=row["Product Name"],
        division=row["Division"],
        region=row["Region"],
        ship_mode=row["Ship Mode"],
        units=row["avg_units"],
        current_factory=row["Factory"],
    )

    best = sim.iloc[0]
    current = sim[sim["Is_Current"]].iloc[0]

    risk = "High" if (row["avg_profit_margin"] < 0.4 and best["Factory"] != row["Factory"]) else "Low"

    recommendations.append({
        "Product Name": row["Product Name"],
        "Region": row["Region"],
        "Ship Mode": row["Ship Mode"],
        "Current Factory": row["Factory"],
        "Recommended Factory": best["Factory"],
        "Current Lead Time": current["Predicted_Lead_Time"],
        "Recommended Lead Time": best["Predicted_Lead_Time"],
        "Lead_Time_Reduction_%": round((1 - best["Predicted_Lead_Time"] / current["Predicted_Lead_Time"]) * 100, 1),
        "Order_Volume": row["order_count"],
        "Avg_Profit_Margin": round(row["avg_profit_margin"], 3),
        "Risk": risk,
        "Action": "Reassign" if best["Factory"] != row["Factory"] else "Keep",
    })

rec_df = pd.DataFrame(recommendations).sort_values("Lead_Time_Reduction_%", ascending=False)
rec_df.head(10)

,Product Name,Region,Ship Mode,Current Factory,Recommended Factory,Current Lead Time,Recommended Lead Time,Lead_Time_Reduction_%,Order_Volume,Avg_Profit_Margin,Risk,Action
114,Wonka Bar -Scrumdiddlyumptious,Gulf,Same Day,Lot's O' Nuts,Wicked Choccy's,1.10,0.41,62.7,12,0.694,Low,Reassign
82,Wonka Bar - Nutty Crunch Surprise,Gulf,Same Day,Lot's O' Nuts,Wicked Choccy's,1.07,0.49,54.2,15,0.713,Low,Reassign
130,Wonka Gum,Gulf,Same Day,Secret Factory,Wicked Choccy's,1.16,0.54,53.4,1,0.520,Low,Reassign
110,Wonka Bar -Scrumdiddlyumptious,Atlantic,Same Day,Lot's O' Nuts,Secret Factory,1.22,0.60,50.8,34,0.694,Low,Reassign
46,Wonka Bar - Fudge Mallows,Atlantic,Same Day,Lot's O' Nuts,Secret Factory,1.22,0.64,47.5,22,0.667,Low,Reassign
78,Wonka Bar - Nutty Crunch Surprise,Atlantic,Same Day,Lot's O' Nuts,Secret Factory,1.23,0.65,47.2,33,0.713,Low,Reassign
50,Wonka Bar - Fudge Mallows,Gulf,Same Day,Lot's O' Nuts,Wicked Choccy's,1.05,0.56,46.7,22,0.667,Low,Reassign
118,Wonka Bar -Scrumdiddlyumptious,Interior,Same Day,Lot's O' Nuts,Secret Factory,0.68,0.38,44.1,15,0.694,Low,Reassign
86,Wonka Bar - Nutty Crunch Surprise,Interior,Same Day,Lot's O' Nuts,Secret Factory,0.85,0.48,43.5,25,0.713,Low,Reassign
54,Wonka Bar - Fudge Mallows,Interior,Same Day,Lot's O' Nuts,Wicked Choccy's,0.87,0.52,40.2,24,0.667,Low,Reassign


In [10]:
# model = pipeline.named_steps["model"]
# preprocessor_fitted = pipeline.named_steps["prep"]

# feature_names = preprocessor_fitted.get_feature_names_out()
# coefs = pd.Series(model.coef_, index=feature_names)

# factory_coefs = coefs[coefs.index.str.contains("Factory")].sort_values()
# factory_coefs

In [11]:
print(f"Total recommendations: {len(rec_df)}")
print(rec_df["Action"].value_counts())
print()
print(rec_df["Risk"].value_counts())
print()
print("Recommended factory distribution (reassignments only):")
print(rec_df[rec_df["Action"] == "Reassign"]["Recommended Factory"].value_counts())

Total recommendations: 141
Action
Reassign    106
Keep         35
Name: count, dtype: int64

Risk
Low     127
High     14
Name: count, dtype: int64

Recommended factory distribution (reassignments only):
Recommended Factory
Secret Factory       39
Wicked Choccy's      31
Lot's O' Nuts        21
The Other Factory    13
Sugar Shack           2
Name: count, dtype: int64


In [12]:
print("Model in use:", bundle["model_name"])

Model in use: Random Forest


In [13]:
avg_dist_by_factory = df.groupby("Factory")["Shipping_Distance_KM"].mean().sort_values()
avg_dist_by_factory

Factory
Secret Factory       1325.164976
The Other Factory    1650.355812
Sugar Shack          1768.263289
Wicked Choccy's      1875.766648
Lot's O' Nuts        2113.001501
Name: Shipping_Distance_KM, dtype: float64

In [14]:
rec_df.to_csv(OUT_PATH, index=False)
print(f"Saved {len(rec_df)} recommendations to {OUT_PATH}")

Saved 141 recommendations to ../outputs/reassignment_recommendations.csv
